# 4.3 Cluster cross validation

In [ ]:
import os
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.generators import ARFFDataGenerator
from src.shadows import ClosestClusterShadow
from src.splitter import ClusterSplitter
from src.orchestrator import RegressionOrchestator
from src.models import RandomForestModel, NeuralNetworkMCDropoutModel, BayesianNeuralNetworkModel
from src.ensemble import EnsembleRegressor
from src.noise import AbsoluteGaussianNoise
from src.interpreter import UncertaintyInterpreter

import warnings
warnings.filterwarnings('ignore')


## 1. Azpiegitura Sortzea (Models, Splitter, etab.)

In [ ]:
datasets = ['housing', 'pbc', 'quake', 'cpu_act', 'cal_housing', 'mv']
data_dir = '../data/weka_arff'

m_rf = RandomForestModel(n_estimators=50, max_depth=10)
m_drop = NeuralNetworkMCDropoutModel(hidden_dim=64, dropout_rate=0.2, epochs=300, n_iter=50)
m_bnn = BayesianNeuralNetworkModel(hidden_dim=32, epochs=300, n_iter=50)
ensemble = EnsembleRegressor(models=[m_rf, m_drop, m_bnn])

models_dict = {
    'RF': m_rf,
    'BNN': m_bnn,
    'Ensemble': ensemble
}

shadow = ClosestClusterShadow(k_range=(4, 8), random_state=42)
splitter = ClusterSplitter(test_size=0.2, random_state=42)
dummy_noise = AbsoluteGaussianNoise(std_dev=0.0)


## 2. Iterazioa

In [ ]:
results = []

for ds_name in datasets:
    print(f"\n==================================================")
    print(f"Datu-multzoa aztertzen: {ds_name}")
    print(f"==================================================")
    
    file_path = os.path.join(data_dir, f"{ds_name}.arff")
    if not os.path.exists(file_path):
        print(f"Ez da aurkitu: {file_path}")
        continue
        
    try:
        generator = ARFFDataGenerator(dataset_name=ds_name, data_dir=data_dir) 
        
        for model_name, model in models_dict.items():
            print(f"  -> Modeloa ebaluatzen: {model_name}")
            orchestrator = RegressionOrchestator(
                generator=generator,
                splitter=splitter,
                model=model
            )
            
            artifacts = orchestrator.run_cv(n_samples=300, x_range=(0,1), noise_strategy=dummy_noise, shadow=shadow)
            interpreter = UncertaintyInterpreter()
            
            mse_v_list, mse_s_list = [], []
            unc_v_list, unc_s_list = [], []
            acc_ood_list = []
            
            for idx, af in enumerate(artifacts):
                uv = af.uncertainties['visible']
                us = af.uncertainties['shadow']
                
                metrics = interpreter.evaluate(uv, us)
                
                mse_v_list.append(af.metrics.mse_test_visible)
                mse_s_list.append(af.metrics.mse_test_shadow)
                unc_v_list.append(np.mean(uv))
                unc_s_list.append(np.mean(us))
                acc_ood_list.append(metrics['accuracy'])
                
            res_dict = {
                'Dataset': ds_name,
                'Model': model_name,
                'MSE_Visible': np.mean(mse_v_list),
                'MSE_Shadow': np.mean(mse_s_list),
                'Unc_Visible': np.mean(unc_v_list),
                'Unc_Shadow': np.mean(unc_s_list),
                'OOD_Accuracy': np.mean(acc_ood_list)
            }
            results.append(res_dict)
            
    except Exception as e:
        print(f"Errorea {ds_name} prozesatzean: {str(e)}")

df_results = pd.DataFrame(results)
df_results['Ignorance_Ratio'] = df_results['Unc_Shadow'] / df_results['Unc_Visible']

display(df_results[['Dataset', 'Model', 'OOD_Accuracy', 'Ignorance_Ratio']])


## 3. Ereduen Ebaluazio Konparatiboa

In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(data=df_results, x='Dataset', y='OOD_Accuracy', hue='Model')
plt.title('RF vs BNN vs Ensemble')
plt.axhline(y=0.5, color='r', linestyle='--', label='Zoria (50%)')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))
sns.barplot(data=df_results, x='Dataset', y='Ignorance_Ratio', hue='Model')
plt.title('Ignorance Ratio')
plt.axhline(y=1.0, color='r', linestyle='--', label='1.0 (Berdintasun muga)')
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.show()
